# Importación de librerias

In [562]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [563]:
import sys
sys.executable


'/Users/carlagallo/Desktop/Proyectos DA/EDA---MARKETING/.venv/bin/python'

In [564]:
import openpyxl
openpyxl.__version__
pd.__version__

'3.0.0'

# Carga de datos #

# 1.Campañas de marketing #

# Columnas del dataset  #

age: La edad del cliente.

job: La ocupación o profesión del cliente.

marital: El estado civil del cliente.

education: El nivel educativo del cliente.

default: Indica si el cliente tiene algún historial de incumplimiento de pagos (1: Sí, 0: No).

housing: Indica si el cliente tiene un préstamo hipotecario (1: Sí, 0: No).

loan: Indica si el cliente tiene algún otro tipo de préstamo (1: Sí, 0: No).

contact: El método de contacto utilizado para comunicarse con el cliente.

duration: La duración en segundos de la última interacción con el cliente.

campaign: El número de contactos realizados durante esta campaña para este cliente.

pdays: Número de días que han pasado desde la última vez que se contactó con el cliente durante esta campaña.

previous: Número de veces que se ha contactado con el cliente antes de esta campaña.

poutcome: Resultado de la campaña de marketing anterior.

emp.var.rate: La tasa de variación del empleo.

cons.price.idx: El índice de precios al consumidor.

cons.conf.idx: El índice de confianza del consumidor.

euribor3m: La tasa de interés de referencia a tres meses.

nr.employed: El número de empleados.

y: Indica si el cliente ha suscrito un producto o servicio (Sí/No).

date: La fecha en la que se realizó la interacción con el cliente.


id_: Un identificador único para cada registro en el dataset.

In [581]:
df_marketing = pd.read_csv('../Datos/raw/bank-additional.csv')

In [ ]:
df_marketing.head()

In [ ]:
df_marketing.shape

In [ ]:
df_marketing.info()

In [ ]:
df_marketing.describe()

In [ ]:
df_marketing.columns

In [582]:
df_marketing.columns = (
    df_marketing.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('.', '_', regex=False)
)
df_marketing.columns

Index(['unnamed:_0', 'age', 'job', 'marital', 'education', 'default',
       'housing', 'loan', 'contact', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx',
       'cons_conf_idx', 'euribor3m', 'nr_employed', 'y', 'date', 'latitude',
       'longitude', 'id_'],
      dtype='str')

In [583]:
df_marketing.rename(columns={'id_': 'id'}, inplace=True)
df_marketing.columns

Index(['unnamed:_0', 'age', 'job', 'marital', 'education', 'default',
       'housing', 'loan', 'contact', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp_var_rate', 'cons_price_idx',
       'cons_conf_idx', 'euribor3m', 'nr_employed', 'y', 'date', 'latitude',
       'longitude', 'id'],
      dtype='str')

# Duplicados

In [ ]:
df_marketing.duplicated().sum()

# Normalización

VALORES NULOS 

In [584]:
df_marketing.isnull().sum()

unnamed:_0           0
age               5120
job                345
marital             85
education         1807
default           8981
housing           1026
loan              1026
contact              0
duration             0
campaign             0
pdays                0
previous             0
poutcome             0
emp_var_rate         0
cons_price_idx     471
cons_conf_idx        0
euribor3m         9256
nr_employed          0
y                    0
date               248
latitude             0
longitude            0
id                   0
dtype: int64

In [585]:
""""mejor visualización de valores nulos para análisis exploratorio"""
df_marketing.isnull().sum()/len(df_marketing)*100

unnamed:_0         0.000000
age               11.906977
job                0.802326
marital            0.197674
education          4.202326
default           20.886047
housing            2.386047
loan               2.386047
contact            0.000000
duration           0.000000
campaign           0.000000
pdays              0.000000
previous           0.000000
poutcome           0.000000
emp_var_rate       0.000000
cons_price_idx     1.095349
cons_conf_idx      0.000000
euribor3m         21.525581
nr_employed        0.000000
y                  0.000000
date               0.576744
latitude           0.000000
longitude          0.000000
id                 0.000000
dtype: float64

Se imputan los valores nulos de variables categóricas con 'unknown' debido a su bajo porcentaje de ausencia, evitando sesgos en el análisis. 

In [586]:
"""Manejo de valores nulos en variables categóricas"""

def categorical_cols(df, columnas):
    for col in columnas:
        df[col] = df[col].fillna('unknown')
    return df
 
categorical_cols_list = [
    'job',
    'marital',
    'education',
    'default',
    'housing',
    'loan'
]


In [587]:
df_marketing = categorical_cols(df_marketing, categorical_cols_list)

In [588]:
df_marketing.isnull().sum()

unnamed:_0           0
age               5120
job                  0
marital              0
education            0
default              0
housing              0
loan                 0
contact              0
duration             0
campaign             0
pdays                0
previous             0
poutcome             0
emp_var_rate         0
cons_price_idx     471
cons_conf_idx        0
euribor3m         9256
nr_employed          0
y                    0
date               248
latitude             0
longitude            0
id                   0
dtype: int64

In [589]:
"""Se completa el valor nulo de la variable 'euribor3m' con el dato faltante que es el mismo en todos los casos"""

df_marketing['euribor3m']= df_marketing ['euribor3m'].fillna(4.857)

In [590]:
"""Se completa el valor nulo de la variable 'age' con la mediana de la misma"""

df_marketing['age'] = df_marketing['age'].fillna(df_marketing['age'].median())

In [591]:
""" Se cambia el formato de la variable 'cons_price_idx' para convertirla a numérica """

df_marketing['cons_price_idx'] = df_marketing['cons_price_idx'].astype(str).str.replace(',', '.').str.strip()

df_marketing['cons_price_idx'] = pd.to_numeric(df_marketing['cons_price_idx'], errors='coerce')

In [592]:
"""Se completa el valor nulo de la variable 'cons_price_idx' con la mediana de la misma"""

df_marketing['cons_price_idx'] = df_marketing['cons_price_idx'].fillna(df_marketing['cons_price_idx'].median())

In [ ]:
df_marketing.isnull().sum()

In [ ]:
"""Tramiento de columna date"""

df_marketing["date"].head()

In [593]:
"""Creación de diccionario de meses"""
meses = {
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12"
}


In [596]:
"""Normalización de texto en columna date"""

df_marketing["date"] = (
    df_marketing["date"]
    .str.lower()
    .str.strip()
)


In [597]:
"""Reemplazo de meses en texto por meses en número"""

for mes_texto, mes_num in meses.items():
    df_marketing["date"] = df_marketing["date"].str.replace(
        f"-{mes_texto}-",
        f"-{mes_num}-",
        regex=False
    )


In [598]:
"""Verificación de cambios en columna date"""

df_marketing["date"].head()

0     2-08-2019
1    14-09-2016
2    15-02-2019
3    29-11-2015
4    29-01-2017
Name: date, dtype: str

In [599]:
"""Convertir columna date a formato fecha"""

df_marketing["date"] = pd.to_datetime(
    df_marketing["date"],
    format="%d-%m-%Y",
    errors="coerce"
)
df_marketing["date"].head()

0   2019-08-02
1   2016-09-14
2   2019-02-15
3   2015-11-29
4   2017-01-29
Name: date, dtype: datetime64[us]

In [600]:
df_marketing["date"].isna().sum()


np.int64(248)

Las fechas nulas no fueron eliminadas ya que pueden reflejar incidencias operativas en la campaña de marketing.
Se creó una variable indicadora para identificar estos casos y se imputó la fecha mediante la mediana únicamente para permitir el análisis temporal, preservando la información original.

In [601]:
def impute_date_with_median(df, date_col):
    """
    Imputa valores nulos en una variable de fecha utilizando la mediana,
    creando previamente una variable indicadora y generando features temporales
    para análisis.
    """
    # Variable indicadora: 1 si la fecha es nula, 0 si no
    df[f'{date_col}_missing'] = df[date_col].isnull().astype(int)

    # Cálculo de la mediana
    fecha_mediana = df[date_col].median()

    # Imputación de la fecha
    df[f'{date_col}_imputed'] = df[date_col].fillna(fecha_mediana)

    # Variables temporales
    df['year'] = df[f'{date_col}_imputed'].dt.year
    df['month'] = df[f'{date_col}_imputed'].dt.month
    df['weekday'] = df[f'{date_col}_imputed'].dt.day_name()

    return df


In [602]:
df_marketing = impute_date_with_median(df_marketing, 'date')
df_marketing[['date', 'date_missing', 'date_imputed', 'year', 'month', 'weekday']].head()

,date,date_missing,date_imputed,year,month,weekday
0,2019-08-02,0,2019-08-02,2019,8,Friday
1,2016-09-14,0,2016-09-14,2016,9,Wednesday
2,2019-02-15,0,2019-02-15,2019,2,Friday
3,2015-11-29,0,2015-11-29,2015,11,Sunday
4,2017-01-29,0,2017-01-29,2017,1,Sunday


Normalizar datos

In [604]:
"""Normalización de datos 1 y 0 por "yes" y "no" """

df_marketing["default"] = df_marketing["default"].replace({1.0: "yes", 0.0: "no"})
df_marketing["housing"] = df_marketing["housing"].replace({1.0: "yes", 0.0: "no"})
df_marketing["loan"] = df_marketing["loan"].replace({1.0: "yes", 0.0: "no"})

In [605]:
"""Normalización de "age"""""

df_marketing["age"] = df_marketing["age"].astype(int)
df_marketing["age"].head()

0    38
1    57
2    37
3    40
4    56
Name: age, dtype: int64

In [606]:
"""Normalización de "education" """

df_marketing["education"] = df_marketing["education"].replace({
    "basic.4y": "basic",
    "basic.6y": "basic",
    "basic.9y": "basic",
    "high.school": "high school",
    "illiterate": "unknown"
})
df_marketing["education"].value_counts()

education
basic                  13051
university.degree      12722
high school             9925
professional.course     5477
unknown                 1825
Name: count, dtype: int64

In [ ]:
df_marketing.head()

In [ ]:
df_marketing.info()
df_marketing.nunique()

Archivo de dataset clean

In [ ]:
df_marketing.to_csv(
    "../Datos/output/df_marketing_clean.csv",
    index=False
)


# 2.Características del cliente #

# Columnas del dataset  #

Income: Representa el ingreso anual del cliente en términos monetarios.

Kidhome: Indica el número de niños en el hogar del cliente.

Teenhome: Indica el número de adolescentes en el hogar del cliente.

Dt_Customer: Representa la fecha en que el cliente se convirtió en cliente de la empresa.

NumWebVisitsMonth: Indica la cantidad de visitas mensuales del cliente al sitio web de la empresa.

ID: Identificador único del cliente.

In [565]:
df_customers = pd.read_excel('../Datos/raw/customer-details.xlsx')

In [566]:
xls = pd.read_excel(
    '../Datos/raw/customer-details.xlsx',
    sheet_name=None
)

In [567]:
""" VISTA DE LAS HOJAS DEL ARCHIVO EXCEL """

xls.keys()

dict_keys(['2012', '2013', '2014'])

In [568]:
df_customers.shape

(20115, 7)

In [569]:
""" CONCATENAR LAS HOJAS DEL ARCHIVO EXCEL EN UN SOLO DATAFRAME """

df_customers = pd.concat(
    xls.values(),
    ignore_index=True
)

In [570]:
df_customers.shape

(43170, 7)

In [571]:
df_customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 43170 entries, 0 to 43169
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Unnamed: 0         43170 non-null  int64         
 1   Income             43170 non-null  int64         
 2   Kidhome            43170 non-null  int64         
 3   Teenhome           43170 non-null  int64         
 4   Dt_Customer        43170 non-null  datetime64[us]
 5   NumWebVisitsMonth  43170 non-null  int64         
 6   ID                 43170 non-null  str           
dtypes: datetime64[us](1), int64(5), str(1)
memory usage: 2.3 MB


In [572]:
df_customers.head()

,Unnamed: 0,Income,Kidhome,Teenhome,Dt_Customer,NumWebVisitsMonth,ID
0,0,161770,1,0,2012-04-04,29,089b39d8-e4d0-461b-87d4-814d71e0e079
1,1,85477,1,1,2012-12-30,7,e9d37224-cb6f-4942-98d7-46672963d097
2,2,147233,1,1,2012-02-02,5,3f9f49b5-e410-4948-bf6e-f9244f04918b
3,3,121393,1,2,2012-12-21,29,9991fafb-4447-451a-8be2-b0df6098d13e
4,4,63164,1,2,2012-06-20,20,eca60b76-70b6-4077-80ba-bc52e8ebb0eb


In [573]:
df_customers.columns

Index(['Unnamed: 0', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer',
       'NumWebVisitsMonth', 'ID'],
      dtype='str')

In [574]:
df_customers.isnull().sum()

Unnamed: 0           0
Income               0
Kidhome              0
Teenhome             0
Dt_Customer          0
NumWebVisitsMonth    0
ID                   0
dtype: int64

In [575]:
df_customers.duplicated().sum()

np.int64(0)

In [576]:
df_customers.columns = (
    df_customers.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('.', '_', regex=False)
)


In [577]:
df_customers.columns

Index(['unnamed:_0', 'income', 'kidhome', 'teenhome', 'dt_customer',
       'numwebvisitsmonth', 'id'],
      dtype='str')

In [613]:
df_customers.to_csv(
    "../Datos/output/df_customers_clean.csv",
    index=False
)


In [578]:
df_customers['id'].dtype

<StringDtype(storage='python', na_value=nan)>

Comprobación de datos previas al cruce de los dataframe

In [579]:
df_marketing['id'].unique()


<StringArray>
['089b39d8-e4d0-461b-87d4-814d71e0e079',
 'e9d37224-cb6f-4942-98d7-46672963d097',
 '3f9f49b5-e410-4948-bf6e-f9244f04918b',
 '9991fafb-4447-451a-8be2-b0df6098d13e',
 'eca60b76-70b6-4077-80ba-bc52e8ebb0eb',
 'd63ede72-0b6d-45b1-8872-385ac6897f65',
 '5e3483e5-236d-437d-8351-541f9d09b9dd',
 '87fdc08b-30ae-4dab-803f-561ecdf27ff0',
 '87b79988-2be5-419d-88f4-56655852c565',
 'ea6b7d04-9271-4c0a-a01f-07795d164aba',
 ...
 '649cf395-b67a-416c-b9ae-3eaf6d3661c5',
 '12d4e85c-39d9-4193-a27d-a58e7af15a43',
 '8b6538f1-e279-4087-8082-659870ab3881',
 '0490dbb9-e21d-4b59-a402-93756e8f17da',
 'd9f2c31c-7623-44df-9240-b4514bf21abd',
 '4eed05de-2a98-4227-b488-32122009b638',
 '0f0aca88-4088-4fe2-905f-44fb675d9493',
 'cadadd4b-7ee5-4019-b13a-ca01bb67ca5b',
 '5f432048-d515-4bb5-9c94-62db451f88d4',
 '993bbbd6-4dbc-4a40-a408-f91f8462bee6']
Length: 43000, dtype: str

In [580]:
df_customers['id'].unique()


<StringArray>
['089b39d8-e4d0-461b-87d4-814d71e0e079',
 'e9d37224-cb6f-4942-98d7-46672963d097',
 '3f9f49b5-e410-4948-bf6e-f9244f04918b',
 '9991fafb-4447-451a-8be2-b0df6098d13e',
 'eca60b76-70b6-4077-80ba-bc52e8ebb0eb',
 'd63ede72-0b6d-45b1-8872-385ac6897f65',
 '5e3483e5-236d-437d-8351-541f9d09b9dd',
 '87fdc08b-30ae-4dab-803f-561ecdf27ff0',
 '87b79988-2be5-419d-88f4-56655852c565',
 'ea6b7d04-9271-4c0a-a01f-07795d164aba',
 ...
 '0490dbb9-e21d-4b59-a402-93756e8f17da',
 'd9f2c31c-7623-44df-9240-b4514bf21abd',
 '4eed05de-2a98-4227-b488-32122009b638',
 '0f0aca88-4088-4fe2-905f-44fb675d9493',
 'cadadd4b-7ee5-4019-b13a-ca01bb67ca5b',
 '5f432048-d515-4bb5-9c94-62db451f88d4',
 '993bbbd6-4dbc-4a40-a408-f91f8462bee6',
 'd6271666-319d-42c8-a741-cb22bf2c2093',
 'f67fbfa8-6573-414d-a805-b26a2f1b1ceb',
 '9727bc06-c11a-461a-a5bb-3d210467cc2a']
Length: 43170, dtype: str

# Cruce de datasframe

- Se observa, a primera vista que hay más clientes que a los registrados en la campaña de MK. 

¿No se ha llamado a la totalidad de los clientes? 

In [612]:
df_marketing_clean = pd.read_csv("../Datos/output/df_marketing_clean.csv")

In [614]:
df_customers_clean = pd.read_csv("../Datos/output/df_customers_clean.csv")

In [615]:
df_resultado = df_customers_clean.merge(
    df_marketing_clean,
    on='id',
    how='outer',
    indicator=True
)


In [616]:
df_resultado.head()

,unnamed:_0_x,income,kidhome,teenhome,dt_customer,numwebvisitsmonth,id,unnamed:_0_y,age,job,...,y,date,latitude,longitude,date_missing,date_imputed,year,month,weekday,_merge
0,12122,101916,2,0,2014-07-17,3,0000e811-006e-4404-b535-89bf6cd96553,36832.0,22.0,services,...,no,2018-01-23,37.753,-110.119,0.0,2018-01-23,2018.0,1.0,Tuesday,both
1,11896,57990,2,2,2014-01-04,3,0000ea53-e9b2-4b3f-9f4b-058f37e5fab8,40976.0,56.0,technician,...,yes,2018-12-02,27.766,-89.350,0.0,2018-12-02,2018.0,12.0,Sunday,both
2,4203,175137,1,1,2014-12-01,8,000165f9-20c0-4cb5-bd47-6233b92655c1,33283.0,31.0,blue-collar,...,no,2015-01-13,36.347,-69.175,0.0,2015-01-13,2015.0,1.0,Tuesday,both
3,2696,62489,2,0,2012-01-25,7,00024507-c59b-4eee-86d5-cc341b96eb6d,2696.0,38.0,blue-collar,...,no,2015-09-02,26.893,-68.620,0.0,2015-09-02,2015.0,9.0,Wednesday,both
4,8810,169187,2,2,2014-05-12,16,0004e1d1-958d-4abf-a57c-9b9c7be887a0,37890.0,39.0,entrepreneur,...,yes,2018-04-30,48.901,-96.742,0.0,2018-04-30,2018.0,4.0,Monday,both
